# Brain MRI — Binary Classifier (Tumor / No Tumor)

Fine-tunes an ImageNet-pretrained **ResNet18** for **binary** brain MRI classification.

| Class | Index |
|-------|-------|
| `Tumor` | 0 |
| `No Tumor` | 1 |

Training uses a 2-class linear head + **`CrossEntropyLoss`** (softmax over logits at inference).

**Outputs**
- Best checkpoint → `backend/model_weights/brain_mri/best_model.pth`
- Metrics JSON → `backend/model_weights/brain_mri/metrics_report.json`
- Accuracy, precision, recall, F1, AUC, confusion matrix

**Expected data layout**
```
data/brain_mri/
  train/{Tumor,No Tumor}/*
  val/{Tumor,No Tumor}/*      # optional — carved from train if missing
  test/{Tumor,No Tumor}/*     # optional — carved from train if missing
```

Folder aliases also accepted: `tumor` / `no_tumor` / `notumor` / `no-tumor`.

## 1. Imports & configuration

In [ ]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "brain_mri"
WEIGHTS_DIR = REPO_ROOT / "backend" / "model_weights" / "brain_mri"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = WEIGHTS_DIR / "best_model.pth"
METRICS_PATH = WEIGHTS_DIR / "metrics_report.json"

# Must match backend/app/models/brain_mri/model.py
CLASS_NAMES = ["Tumor", "No Tumor"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.15
TEST_RATIO = 0.15
NUM_WORKERS = 0
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Data dir: {DATA_DIR}")
print(f"Checkpoint: {BEST_MODEL_PATH}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

## 2. Transforms (flip, rotation, brightness)

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.25, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

## 3. Dataset (folder-per-class)

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

FOLDER_ALIASES = {
    "tumor": "Tumor",
    "no tumor": "No Tumor",
    "no_tumor": "No Tumor",
    "notumor": "No Tumor",
    "no-tumor": "No Tumor",
    "healthy": "No Tumor",
    "normal": "No Tumor",
}


def normalize_class_folder(name: str) -> str | None:
    key = name.strip().lower().replace("  ", " ")
    if name in CLASS_NAMES:
        return name
    return FOLDER_ALIASES.get(key)


def collect_folder_samples(root: Path, split: str) -> list[tuple[Path, int]]:
    split_dir = root / split
    samples: list[tuple[Path, int]] = []
    if not split_dir.is_dir():
        return samples
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label_name = normalize_class_folder(class_dir.name)
        if label_name is None:
            continue
        label_idx = CLASS_NAMES.index(label_name)
        for p in class_dir.rglob("*"):
            if p.suffix.lower() in IMAGE_EXTS:
                samples.append((p, label_idx))
    return samples


class BrainMRIDataset(Dataset):
    def __init__(self, samples: list[tuple[Path, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


print("Dataset helpers ready.")

## 4. Bootstrap demo data (optional)

If `data/brain_mri` is empty, generate a small synthetic set so the pipeline runs end-to-end. Replace with a real MRI corpus for meaningful metrics.

In [ ]:
def _synthetic_mri(seed: int, has_tumor: bool, size: int = 256) -> Image.Image:
    rng = random.Random(seed)
    img = Image.new("L", (size, size), color=12 + rng.randint(0, 10))
    draw = ImageDraw.Draw(img)
    # skull + brain oval
    draw.ellipse((30, 25, 226, 235), fill=40 + rng.randint(0, 15))
    draw.ellipse((55, 50, 201, 210), fill=90 + rng.randint(0, 25))
    if has_tumor:
        cx = rng.randint(90, 160)
        cy = rng.randint(80, 160)
        r = rng.randint(18, 36)
        draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=180 + rng.randint(0, 40))
        # edema ring
        draw.ellipse((cx - r - 8, cy - r - 8, cx + r + 8, cy + r + 8), outline=140, width=3)
    img = img.filter(ImageFilter.GaussianBlur(radius=1.0))
    img = ImageEnhance.Contrast(img).enhance(1.2)
    return img.convert("RGB")


def bootstrap_synthetic_dataset(n_per_class: int = 40) -> None:
    idx = 0
    for label_name, has_tumor in [("Tumor", True), ("No Tumor", False)]:
        for i in range(n_per_class):
            r = (idx % 10) / 10.0
            split = "train" if r < 0.7 else "val" if r < 0.85 else "test"
            out_dir = DATA_DIR / split / label_name
            out_dir.mkdir(parents=True, exist_ok=True)
            path = out_dir / f"{label_name.replace(' ', '_').lower()}_{i:03d}.png"
            _synthetic_mri(seed=idx + 11, has_tumor=has_tumor).save(path)
            idx += 1
    print(f"Wrote synthetic brain MRI samples under {DATA_DIR}")


existing = collect_folder_samples(DATA_DIR, "train")
if len(existing) < 8:
    print("Few/no training images found — generating synthetic demo set…")
    bootstrap_synthetic_dataset()
else:
    print(f"Found {len(existing)} train samples — skipping bootstrap.")

## 5. Train / val / test splits

In [ ]:
train_samples = collect_folder_samples(DATA_DIR, "train")
val_samples = collect_folder_samples(DATA_DIR, "val")
test_samples = collect_folder_samples(DATA_DIR, "test")

if not val_samples or not test_samples:
    pool = list(train_samples)
    if not pool:
        raise RuntimeError(f"No samples under {DATA_DIR}. Add images or re-run bootstrap.")
    rng = random.Random(SEED)
    rng.shuffle(pool)
    n = len(pool)
    n_test = max(1, int(n * TEST_RATIO)) if not test_samples else 0
    n_val = max(1, int(n * VAL_RATIO)) if not val_samples else 0
    cursor = 0
    if not test_samples:
        test_samples = pool[cursor : cursor + n_test]
        cursor += n_test
    if not val_samples:
        val_samples = pool[cursor : cursor + n_val]
        cursor += n_val
    train_samples = pool[cursor:] or pool

train_ds = BrainMRIDataset(train_samples, transform=train_transform)
val_ds = BrainMRIDataset(val_samples, transform=eval_transform)
test_ds = BrainMRIDataset(test_samples, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


def count_by_class(samples: list[tuple[Path, int]]) -> dict[str, int]:
    counts = {c: 0 for c in CLASS_NAMES}
    for _, y in samples:
        counts[CLASS_NAMES[y]] += 1
    return counts


print(f"Train: {len(train_ds)} {count_by_class(train_samples)}")
print(f"Val:   {len(val_ds)} {count_by_class(val_samples)}")
print(f"Test:  {len(test_ds)} {count_by_class(test_samples)}")

## 6. Model — ResNet18 + 2-class head

Outputs **logits**; Softmax is applied for probabilities. Loss = `CrossEntropyLoss`.

In [ ]:
def build_resnet18(num_classes: int = NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    try:
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
    except Exception:
        model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


model = build_resnet18(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")
print(model.fc)

## 7. Training helpers

In [ ]:
def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss = 0.0
    n = 0
    all_logits = []
    all_targets = []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, targets in tqdm(loader, leave=False):
            images = images.to(device)
            targets = targets.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            if train:
                loss.backward()
                optimizer.step()
            bs = images.size(0)
            total_loss += loss.item() * bs
            n += bs
            all_logits.append(logits.detach().cpu())
            all_targets.append(targets.detach().cpu())

    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets).numpy()
    probs = F.softmax(logits, dim=1).numpy()
    preds = probs.argmax(axis=1)
    return total_loss / max(n, 1), probs, preds, targets


def binary_metrics(probs: np.ndarray, preds: np.ndarray, targets: np.ndarray) -> dict:
    # Positive class = Tumor (index 0) for precision/recall/F1/AUC
    pos_prob = probs[:, 0]
    metrics = {
        "accuracy": float(accuracy_score(targets, preds)),
        "precision": float(precision_score(targets, preds, pos_label=0, zero_division=0)),
        "recall": float(recall_score(targets, preds, pos_label=0, zero_division=0)),
        "f1": float(f1_score(targets, preds, pos_label=0, zero_division=0)),
        "macro_f1": float(f1_score(targets, preds, average="macro", zero_division=0)),
    }
    if len(np.unique(targets)) > 1:
        metrics["auc"] = float(roc_auc_score(targets == 0, pos_prob))
    else:
        metrics["auc"] = float("nan")
    metrics["confusion_matrix"] = confusion_matrix(targets, preds, labels=[0, 1]).tolist()
    return metrics


print("Training helpers ready.")

## 8. Train with checkpointing

Best model selected by **validation AUC** (falls back to F1 if AUC is undefined).

In [ ]:
history = []
best_score = -1.0
best_epoch = -1
t0 = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_probs, train_preds, train_tgts = run_epoch(model, train_loader, optimizer)
    val_loss, val_probs, val_preds, val_tgts = run_epoch(model, val_loader, optimizer=None)
    scheduler.step()

    train_m = binary_metrics(train_probs, train_preds, train_tgts)
    val_m = binary_metrics(val_probs, val_preds, val_tgts)
    score = val_m["auc"]
    if np.isnan(score):
        score = val_m["f1"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_m["accuracy"],
        "val_acc": val_m["accuracy"],
        "val_f1": val_m["f1"],
        "val_auc": val_m["auc"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f}  "
        f"val_acc={val_m['accuracy']:.3f} val_f1={val_m['f1']:.3f} val_auc={val_m['auc']:.3f}"
    )

    if score > best_score:
        best_score = score
        best_epoch = epoch
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "architecture": "resnet18",
            "class_names": CLASS_NAMES,
            "num_classes": NUM_CLASSES,
            "multi_label": False,
            "scan_type": "brain_mri",
            "image_size": IMAGE_SIZE,
            "epoch": epoch,
            "val_auc": val_m["auc"],
            "val_f1": val_m["f1"],
            "val_accuracy": val_m["accuracy"],
            "val_metrics": val_m,
        }
        torch.save(checkpoint, BEST_MODEL_PATH)
        print(f"  ↳ saved best → {BEST_MODEL_PATH} (score={score:.4f})")

elapsed = time.perf_counter() - t0
print(f"\nTraining done in {elapsed/60:.1f} min | best epoch={best_epoch} score={best_score:.4f}")

## 9. Test evaluation (accuracy, P/R/F1, AUC, confusion matrix)

In [ ]:
ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_loss, test_probs, test_preds, test_tgts = run_epoch(model, test_loader, optimizer=None)
test_metrics = binary_metrics(test_probs, test_preds, test_tgts)
cm = np.array(test_metrics["confusion_matrix"])

print(f"Test loss: {test_loss:.4f}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}  (positive=Tumor)")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1:        {test_metrics['f1']:.4f}")
print(f"AUC:       {test_metrics['auc']:.4f}\n")
print(classification_report(test_tgts, test_preds, target_names=CLASS_NAMES, zero_division=0))
print("Confusion matrix [rows=true, cols=pred]:")
print(cm)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1], CLASS_NAMES)
axes[0].set_yticks([0, 1], CLASS_NAMES)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_title("Confusion matrix")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, int(cm[i, j]), ha="center", va="center", color="black")
fig.colorbar(im, ax=axes[0], fraction=0.046)

if len(np.unique(test_tgts)) > 1:
    RocCurveDisplay.from_predictions(test_tgts == 0, test_probs[:, 0], ax=axes[1], name="Tumor")
    axes[1].set_title("ROC curve")
else:
    axes[1].text(0.5, 0.5, "ROC needs both classes", ha="center")
    axes[1].axis("off")
plt.tight_layout()
plt.show()

report = {
    "architecture": "resnet18",
    "scan_type": "brain_mri",
    "class_names": CLASS_NAMES,
    "multi_label": False,
    "best_epoch": best_epoch,
    "best_val_score": best_score,
    "test_loss": test_loss,
    "test_metrics": test_metrics,
    "history": history,
    "checkpoint": str(BEST_MODEL_PATH),
}
METRICS_PATH.write_text(json.dumps(report, indent=2))
print(f"\nWrote metrics → {METRICS_PATH}")
print(f"Best model → {BEST_MODEL_PATH}")

## 10. Quick inference smoke test

In [ ]:
@torch.no_grad()
def predict_brain_mri(image_path: Path) -> dict:
    image = Image.open(image_path).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(device)
    logits = model(tensor)
    probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    idx = int(np.argmax(probs))
    return {
        "label": CLASS_NAMES[idx],
        "confidence": float(probs[idx]),
        "probabilities": {CLASS_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)},
    }


sample_path, sample_y = test_samples[0]
print("Sample:", sample_path)
print("True:  ", CLASS_NAMES[sample_y])
print("Pred:  ", predict_brain_mri(sample_path))